# 📤 Terminal 3 — Kafka Producer
Reads rows from hour.csv and sends 1 row/second to the `raw-data` Kafka topic.

**Run this LAST** (after the Streams Processor and Consumer are already running).

In [ ]:
!pip install confluent-kafka pandas --quiet

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ════════════════════════════════════════════════
#   FILL IN YOUR CONFLUENT CLOUD CREDENTIALS HERE
# ════════════════════════════════════════════════
BOOTSTRAP_SERVERS = 'pkc-XXXXX.us-east-1.aws.confluent.cloud:9092'  # <-- replace
SASL_USERNAME     = 'YOUR_API_KEY'                                    # <-- replace
SASL_PASSWORD     = 'YOUR_API_SECRET'                                 # <-- replace
DATASET_PATH      = '/content/drive/MyDrive/kafka-assignment/hour.csv'
RAW_TOPIC         = 'raw-data'
DELAY_SECONDS     = 1.0   # 1 row per second
# ════════════════════════════════════════════════

In [ ]:
import pandas as pd
import json
import time
from confluent_kafka import Producer

# Kafka producer config
conf = {
    'bootstrap.servers': BOOTSTRAP_SERVERS,
    'security.protocol': 'SASL_SSL',
    'sasl.mechanisms':   'PLAIN',
    'sasl.username':     SASL_USERNAME,
    'sasl.password':     SASL_PASSWORD,
}

producer = Producer(conf)
print('Producer connected to Kafka.')

def delivery_report(err, msg):
    if err:
        print(f'[ERROR] Delivery failed: {err}')
    else:
        print(f'[SENT]  Row → topic={msg.topic()} | offset={msg.offset()}')

# Load dataset
df = pd.read_csv(DATASET_PATH)
print(f'Dataset loaded: {len(df)} rows. Starting stream...\n')
print('─' * 60)

# Stream rows one by one
for i, row in df.iterrows():
    record = row.to_dict()
    record['row_index'] = int(i)
    record['timestamp'] = time.time()

    # Convert numpy types to native Python for JSON serialisation
    record = {k: (int(v) if hasattr(v, 'item') and isinstance(v.item(), int)
                  else float(v) if hasattr(v, 'item') else v)
              for k, v in record.items()}

    producer.produce(
        topic=RAW_TOPIC,
        key=str(i),
        value=json.dumps(record),
        callback=delivery_report
    )
    producer.poll(0)  # trigger delivery callbacks
    time.sleep(DELAY_SECONDS)

producer.flush()
print('\n[DONE] All rows sent.')